In [ ]:
%load_ext autoreload
%autoreload 2

import json
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## Process mave db metadata

Source of MaveDB data: https://zenodo.org/records/15653325

### Explore the metadata

In [ ]:
with open('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/main.json') as f:
    d = json.load(f)
d

In [ ]:
d['experimentSets'][1]

In [ ]:
d['experimentSets'][0]['experiments'][0]['scoreSets'][0]

### Process the metadata

In [ ]:
import pandas as pd

# Suppose your full list is called `data`
# Example: data = [ {...}, {...}, ... ]

with open('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/main.json') as f:
    d = json.load(f)
    
rows = []

for item in d['experimentSets']:
    urn = item.get('urn')
    published_date = item.get('publishedDate')
    set_id = item.get('id')
    record_type = item.get('recordType')

    # Loop over experiments in this experiment set
    for exp in item.get('experiments', []):
        exp_title = exp.get('title')
        exp_short_desc = exp.get('shortDescription')
        exp_abstract = exp.get('abstractText')
        exp_method = exp.get('methodText')
        exp_urn = exp.get('urn')
        exp_creation_date = exp.get('creationDate')

        # Loop over scoreSets
        for score in exp.get('scoreSets', []):
            score_title = score.get('title')
            num_variants = score.get('numVariants')
            score_urn = score.get('urn')
            license_name = score.get('license', {}).get('longName')
            
            target_genes = score.get('targetGenes', [])
            # Loop over target genes
            for gene in target_genes:
                gene_name = gene.get('name')
                category = gene.get('category')

                organism_name = None
                target_sequence = gene.get('targetSequence')
                if target_sequence:
                    taxonomy = target_sequence.get('taxonomy')
                    if taxonomy:
                        organism_name = taxonomy.get('organismName')
            
                # Extract external IDs
                ensembl_id = None
                refseq_id = None
                uniprot_id = None

                for ext_id in gene.get('externalIdentifiers', []):
                    identifier = ext_id.get('identifier', {})
                    db_name = identifier.get('dbName', '').lower()
                    id_value = identifier.get('identifier')

                    if db_name == 'ensembl':
                        ensembl_id = id_value
                    elif db_name == 'refseq':
                        refseq_id = id_value
                    elif db_name == 'uniprot':
                        uniprot_id = id_value

                rows.append({
                    'set_urn': urn,
                    'set_published_date': published_date,
                    'set_id': set_id,
                    'record_type': record_type,

                    'experiment_title': exp_title,
                    'experiment_short_desc': exp_short_desc,
                    'experiment_abstract': exp_abstract,
                    'experiment_method': exp_method,
                    'experiment_urn': exp_urn,
                    'experiment_creation_date': exp_creation_date,

                    'score_title': score_title,
                    'num_variants': num_variants,
                    'score_urn': score_urn,
                    'license_name': license_name,

                    'target_gene': gene_name,
                    'category': category,
                    'ensembl_id': ensembl_id,
                    'refseq_id': refseq_id,
                    'uniprot_id': uniprot_id,
                    'organism_name': organism_name
                })

# Build DataFrame
mave_db = pl.DataFrame(rows)

# Extract gene symbols
# mave_db = mave_db.with_columns(
#     pl.col('target_gene').str.split(' ').list.get(0).alias('gene_symbol')
# )

mave_db = mave_db.with_columns(
    pl.col('target_gene')
    .str.split(' ')
    .list.get(0)
    .str.to_uppercase()
    .alias('gene_symbol')
)

mave_db

In [ ]:
mave_db['category'].value_counts().sort('count', descending=True)

In [ ]:
mave_db['organism_name'].value_counts().sort('count', descending=True)

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['target_gene'].value_counts().sort('count', descending=True)

In [ ]:
# Merge to get ENSEMBLE gene ids

dgid = pl.read_parquet('/s/project/deeprvat/deeprvat_input/protein_coding_genes.parquet').rename({'gene_name':'gene_symbol'})

dgid = dgid.with_columns(
    pl.col('gene').str.split('.').list.get(0).alias('gene_id')
).drop(['__index_level_0__', 'gene_type', 'id', 'gene'])

dgid

In [ ]:
mave_db = mave_db.join(dgid, on='gene_symbol', how='left')
mave_db

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_id'].value_counts().sort('count', descending=True)

In [ ]:
tmp = mave_db.filter(pl.col('organism_name')=='Homo sapiens').filter(pl.col('category')=='protein_coding').filter(pl.col('gene_id').is_null())['target_gene', 'gene_symbol', 'set_urn'].unique().sort(by='target_gene')
tmp

In [ ]:
# Dictionary to map the ambiguous target genes

map_dict = {
    'AID': 'AICDA',
    'ARK2C Zinc finger, RING-type domain': 'ARK2C',
    'Aβ42': 'APP',
    'COMT_ROI1_2': 'COMT',
    'DUX4': 'DUX4',
    'GB1': 'IGBP1',
    'GRLF1 FF domain': 'ARHGAP35',
    'Glycophorin A': 'GYPA',
    'IGHG1': 'IGHG1',
    'NA Transcription factor IIS, N-terminal domain': 'TCEA1',
    'NA Ubiquitin-like domain': 'UBL3',
    'PSD95 PDZ3': 'DLG4',
    'RAF': 'RAF1',
    'Ras': 'KRAS',
    'S505N MPL': 'MPL',
    'S505N MPL': 'MPL',
    'SMN Tudor domain': 'SMN',
    'VKOR': 'VKORC1',
    'W515K MPL': 'MPL',
    'alpha-synuclein': 'SNCA',
    'hYAP65 WW domain': 'YAP65',
    'human L-Selectin': 'SELL',
    'p53': 'TP53',
}

# Convert to pandas
df_pd = mave_db.filter(pl.col('target_gene').is_in(map_dict.keys())).to_pandas()

# Map only if key in map_dict, else keep original value
df_pd['gene_symbol'] = df_pd['target_gene'].apply(
    lambda x: map_dict[x]
)

df_pd['gene_symbol'].value_counts().sort_values(ascending=False)

In [ ]:
# Append back to mave_db
sub_mave_db = mave_db.filter(~pl.col('target_gene').is_in(map_dict.keys()))

mave_db = pl.concat([sub_mave_db, pl.from_pandas(df_pd)])
mave_db

In [ ]:
mave_db = mave_db.drop('gene_id').join(dgid, on='gene_symbol', how='left')
mave_db

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens').filter(pl.col('category')=='protein_coding').filter(pl.col('gene_id').is_null())['target_gene', 'gene_symbol', 'set_urn'].unique().sort(by='target_gene')

In [ ]:
gene_id_map_dict = {
    'ARK2C': 'ENSG00000141622',
    'DUX4': 'ENSG00000260596',
    'IGHG1': 'ENSG00000211896',
    'SMN': 'ENSG00000172062',
    'YAP65': 'ENSG00000137693'
}

# Convert to pandas
df_pd = mave_db.filter(pl.col('gene_symbol').is_in(gene_id_map_dict.keys())).to_pandas()

# Map only if key in map_dict, else keep original value
df_pd['gene_id'] = df_pd['gene_symbol'].apply(
    lambda x: gene_id_map_dict[x]
)

# print(df_pd['gene_id'].value_counts().sort_values(ascending=False))

# Append back to mave_db
sub_mave_db = mave_db.filter(~pl.col('gene_symbol').is_in(gene_id_map_dict.keys()))

mave_db = pl.concat([sub_mave_db, pl.from_pandas(df_pd)])
mave_db

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_id'].value_counts().sort('count', descending=True)

In [ ]:
# mave_db.write_csv('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mavedb_metadata.tsv', separator='\t')

## Check gene intersection

In [ ]:
gb_res = pd.read_parquet('/s/project/deeprvat/ukb_gym/genebass/genebass_all_associations_p_e-5.parquet')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

gb_res = pl.from_pandas(gb_res.query("(significant == True) & (modifier != 'custom') & (('pLoF' in annotation) | ('missense' in annotation)) & (trait_type == 'continuous')"))
gb_res

In [ ]:
gb_res.filter(pl.col('gene_symbol')=='PRKN')

In [ ]:
mave_db = pl.read_csv('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mavedb_metadata.tsv', separator='\t')
# mave_db
mgs = mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_id'].value_counts()
mgs

In [ ]:
pg = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/protein_gym/proteingym_SNP_DMS_scores.parquet')
pgs = pg['gene_id'].value_counts()
pgs

In [ ]:
dms_genes = pl.DataFrame({
    'gene_id': list(set(mgs['gene_id']).union(set(pgs['gene_id'])))
})
# dms_genes.write_csv('/s/project/deeprvat/ukb_gym/experimental_assays/dms_genes.txt', include_header=False)

In [ ]:
set(pgs['gene_id']) - set(mgs['gene_id'])

In [ ]:
len(set(gb_res['gene_id']).intersection(set(dms_genes['gene_id'])))

In [ ]:
mgb_genes = mave_db.filter(pl.col('gene_id').is_in(gb_res['gene_id'].unique()))['gene_symbol', 'gene_id'].unique()
mgb_genes

## Variant identifier mapping

### Consolidate all the MAVE scores

In [ ]:
mave_db = pl.read_csv('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mavedb_metadata.tsv', separator='\t')

mave_filt = mave_db.sort('num_variants', descending=True).filter(
    (pl.col('organism_name')=="Homo sapiens") &
    (pl.col('category')=="protein_coding")
)
mave_filt

In [ ]:
mave_filt['num_variants'].sum()

In [ ]:
mave_filt['score_urn', 'gene_id', 'gene_symbol', 'target_gene'].unique()

In [ ]:
mave_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/mave_db'
id_cols = ['score_urn', 'gene_id', 'gene_symbol', 'target_gene']

# Define the core columns to select from each score CSV
core_cols = ['accession', 'hgvs_nt', 'hgvs_splice', 'hgvs_pro', 'score']

var_score_list = []

req_mave_df = mave_filt.select(id_cols).unique()

for row in tqdm(req_mave_df.iter_rows(named=True), total=req_mave_df.height):
    score_urn_safe = row['score_urn'].replace(":", "-")
    csv_path = f"{mave_dir}/csv/{score_urn_safe}.scores.csv"

    tmp = pl.read_csv(csv_path, null_values="NA")

    # Find possible extra column: anything not in core
    extra_cols = [col for col in tmp.columns if col not in core_cols]

    if extra_cols:
        raw_extra_col = extra_cols[0]
        # Sanitize: lower, replace - or : with _, strip whitespace
        extra_col_clean = (
            raw_extra_col.lower()
            .replace("-", "_")
            .replace(":", "_")
            .strip()
        )
    else:
        raw_extra_col = None
        extra_col_clean = None

    if raw_extra_col:
        tmp = tmp.with_columns([
            pl.col(raw_extra_col).alias('extra_col_value'),
            pl.lit(extra_col_clean).alias('extra_col_type')
        ])
    else:
        tmp = tmp.with_columns([
            pl.lit(None).alias('extra_col_value'),
            pl.lit(None).alias('extra_col_type')
        ])

    tmp = tmp.with_columns(
        pl.col('extra_col_value').cast(pl.Float32, strict=False),
        pl.col('extra_col_type').cast(pl.String, strict=False)  # <- force String
    )
    
    tmp = tmp.with_columns(
        pl.col('score').cast(pl.Float32, strict=False),
        pl.col('extra_col_value').cast(pl.Float32, strict=False),
        pl.lit(row['score_urn']).alias('score_urn'),
        pl.lit(row['gene_id']).alias('gene_id'),
        pl.lit(row['gene_symbol']).alias('gene_symbol'),
        pl.lit(row['target_gene']).alias('target_gene')
    ).select(
        id_cols + core_cols + ['extra_col_type', 'extra_col_value']
    )

    var_score_list.append(tmp)

mave_var_scores = pl.concat(var_score_list)
# mave_var_scores.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mave_scores_genebass_genes.parquet')
mave_var_scores

### Amino acid mapping

In [ ]:
# mave_var_scores = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mave_scores_genebass_genes.parquet')
mave_var_scores

In [ ]:
# Example counts
vc = (
    mave_var_scores
    .filter(pl.col('hgvs_pro').is_not_null())
    ['gene_symbol']
    .value_counts()
    .sort('count', descending=True)
).to_pandas()

# Make gene_symbol a categorical with sorted levels
vc['gene_symbol'] = pd.Categorical(vc['gene_symbol'], categories=vc['gene_symbol'], ordered=True)

# Plot
(
    ggplot(vc, aes(x='gene_symbol', y='count')) +
    geom_col() +
    theme_bw() +
    scale_y_log10() +
    annotation_logticks(sides='l') +
    theme(
        axis_text_x=element_text(angle=90),
        figure_size=(8, 4)
    )
)

In [ ]:
aa_3to1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D",
    "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G",
    "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
    "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S",
    "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
    "Ter": "*"   # Sometimes for stop codon
}

# Extract parts
df = mave_var_scores[['hgvs_pro']].with_columns([
    pl.col("hgvs_pro").str.extract(r"p\.([A-Za-z]+)", 1).alias("ref_aa_3"),
    pl.col("hgvs_pro").str.extract(r"p\.[A-Za-z]+(\d+)", 1).alias("pos"),
    pl.col("hgvs_pro").str.extract(r"p\.[A-Za-z]+\d+([A-Za-z]+)", 1).alias("alt_aa_3"),
])

df = df.with_columns(
    pl.when(pl.col("alt_aa_3").str.starts_with("delins"))
    .then(pl.col("alt_aa_3").str.replace("^delins", ""))
    .otherwise(pl.col("alt_aa_3"))
    .alias("alt_aa_3")
)

df
# df.drop_nulls()

In [ ]:
df.drop_nulls().unique()

In [ ]:
# Map 3-letter to 1-letter using map_elements
df = df.drop_nulls().with_columns([
    pl.col("ref_aa_3").map_elements(
        lambda aa: aa_3to1.get(aa, aa),
        return_dtype=pl.Utf8
    ).alias("ref_aa"),
    pl.col("alt_aa_3").map_elements(
        lambda aa: aa_3to1.get(aa, aa),
        return_dtype=pl.Utf8
    ).alias("alt_aa"),
])

# Recombine to single-letter notation
df_fin = df.with_columns([
    pl.format("{}{}{}", pl.col("ref_aa"), pl.col("pos"), pl.col("alt_aa")).alias("mutant_short")
]).select(['hgvs_pro', 'mutant_short']).unique()

df_fin

### Complex variants that couldn't be mapped easily
In some cases teh authors submit one missense variants but repeated many times, we will extract these.

In [ ]:
# Find complex variants that could not be simplified to single-letter notation
cx_vars = mave_var_scores.join(df_fin, on='hgvs_pro', how='left').filter(pl.col('mutant_short').is_null()).select(['hgvs_pro']).unique().sort('hgvs_pro')
cx_vars

In [ ]:
df = (
    cx_vars
    .with_columns(
        pl.col("hgvs_pro")
        .str.replace("p.", "")
        .str.split(":", inclusive=False)
        .list.last()
        .str.strip_prefix("[")
        .str.strip_suffix("]")
        .str.split(";")
        .alias("hgvs_pro_split")
    )
    .explode("hgvs_pro_split")
)
df

In [ ]:
# Add a flag: is_synonymous
df = df.with_columns(
    (pl.col("hgvs_pro_split") == "=").alias("is_synonymous")
)

# Group by original hgvs_pro to check if all variants are identical
keep_df = (
    df.group_by("hgvs_pro")
    .agg([
        pl.col("hgvs_pro_split").unique().alias("unique_vars"),
        pl.col("hgvs_pro_split").filter(pl.col("hgvs_pro_split") != "=").unique().alias("unique_non_synonymous")
    ])
    .filter(
        (pl.col("unique_vars").list.len() == 1) |
        (pl.col("unique_non_synonymous").list.len() == 1)  # If only one unique *non-synonymous*
    )
)

# Convert list to string afterwards
keep_df = keep_df.with_columns(
    pl.col("unique_non_synonymous").list.first().alias("unique_non_synonymous")
)
keep_df

In [ ]:
col2filter = "unique_non_synonymous"

df = keep_df.with_columns([
    (
        pl.when(pl.col(col2filter).str.contains(r"^[A-Za-z]{3}\d+[A-Za-z]{3}$"))
        .then(pl.lit("substitution"))
        .when(pl.col(col2filter).str.contains(r"="))
        .then(pl.lit("synonymous"))
        .when(pl.col(col2filter).str.contains(r"ins"))
        .then(pl.lit("insertion"))
        .otherwise(pl.lit("other"))
        .alias("hgvs_type")
    )
])
df

In [ ]:
df = df.with_columns([
    pl.when(pl.col("hgvs_type") == "substitution")
      .then(pl.col(col2filter).str.extract(r"([A-Za-z]+)", 1))
      .otherwise(None)
      .alias("ref_aa_3"),
    pl.when(pl.col("hgvs_type") == "substitution")
      .then(pl.col(col2filter).str.extract(r"([0-9]+)", 1))
      .otherwise(None)
      .alias("pos"),
    pl.when(pl.col("hgvs_type") == "substitution")
      .then(pl.col(col2filter).str.extract(r"[0-9]+([A-Za-z]+)", 1))
      .otherwise(None)
      .alias("alt_aa_3"),
])

df = df.with_columns(
    pl.when(pl.col("alt_aa_3").str.starts_with("delins"))
    .then(pl.col("alt_aa_3").str.replace("^delins", ""))
    .otherwise(pl.col("alt_aa_3"))
    .alias("alt_aa_3")
)

df

In [ ]:
df_fin_cx = df.with_columns([
    pl.when(pl.col("hgvs_type") == "substitution")
      .then(pl.col("ref_aa_3").map_elements(lambda aa: aa_3to1.get(aa) if aa else None, return_dtype=pl.Utf8))
      .otherwise(None)
      .alias("ref_aa"),
    pl.when(pl.col("hgvs_type") == "substitution")
      .then(pl.col("alt_aa_3").map_elements(lambda aa: aa_3to1.get(aa) if aa else None, return_dtype=pl.Utf8))
      .otherwise(None)
      .alias("alt_aa"),
]).with_columns([
    pl.format("{}{}{}", pl.col("ref_aa"), pl.col("pos"), pl.col("alt_aa")).alias("mutant_short")
]).select(['hgvs_pro', 'mutant_short']).unique()

df_fin_cx

In [ ]:
df_map = pl.concat([df_fin, df_fin_cx]).unique()
df_map

In [ ]:
mave_var_merged = mave_var_scores.join(df_map, on='hgvs_pro', how='left')
mave_var_merged

In [ ]:
mave_var_merged.filter(pl.col('mutant_short').is_null()).select(['hgvs_pro']).unique().sort('hgvs_pro')

In [ ]:
mave_var_merged = mave_var_merged.with_columns(
    pl.col('gene_symbol').alias('gene_name')
).drop('gene_symbol')
mave_var_merged

In [ ]:
# mave_var_merged.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/250717_mavedb_DMS_scores_human_coding_genes.parquet')

## Plot DMS scores

In [ ]:
mave_dms = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/250717_mavedb_DMS_scores_human_coding_genes.parquet').filter(~pl.col('hgvs_pro').is_null())

mave_dms
# mave_dms['gene_name'].value_counts().sort('count', descending=True).head(10)

In [ ]:
a = mave_dms.group_by('score_urn').agg(
    pl.col('score').mean().alias('dms_score_mean'),
    pl.col('score').median().alias('dms_score_median'),
    pl.col('score').std().alias('dms_score_se'),
)

plt_df = mave_dms.join(a, on='score_urn').sort('dms_score_median', descending=True)
plt_df

In [ ]:
(
    ggplot(plt_df, aes(x='score_urn', y='score')) +
    geom_boxplot() +
    theme_bw() +
    theme(
        axis_ticks_x=element_blank(),
        axis_text_x=element_blank(),
        figure_size=(16, 8)
    )
)

In [ ]:
dup_genes = plt_df[['score_urn', 'gene_name']].unique().filter(pl.col('gene_name').is_duplicated())
plt_dup = plt_df.filter(pl.col('gene_name').is_in(dup_genes['gene_name']))
plt_dup

In [ ]:
medians = (
    plt_dup
    .group_by(['gene_name', 'score_urn'])
    .agg(pl.col('score').median().alias('median_score'))
)

median_diffs = (
    medians
    .group_by('gene_name')
    .agg([
        (pl.col('median_score').max() - pl.col('median_score').min()).alias('median_diff'),
        pl.col('score_urn').n_unique().alias('num_exp')
    ])
    .sort('median_diff', descending=True)
)


median_diffs.head(10)

In [ ]:
(
    ggplot(plt_df.filter(pl.col('gene_name').is_in(median_diffs.head(5)['gene_name'])), aes(x='gene_name', y='score', fill='score_urn')) +
    geom_boxplot() +
    theme_bw() +
    theme(
        axis_text_x=element_text(rotation=90),
        figure_size=(14, 7),
        legend_position='none'
    )
)

In [ ]:
plt_df.filter(pl.col('gene_name')=='TP53')

In [ ]:
dup_plt = plt_df.filter(pl.col('gene_name')=='TP53').pivot(
    index='mutant_short',
    columns='score_urn',
    values='score'
).drop_nulls()

print(dup_plt.median())
dup_plt

In [ ]:
mave_var_merged = mave_var_merged.with_columns(
    (pl.col('gene_name') + '_' + pl.col('mutant_short')).alias('gene_mutant')
)

reps = mave_var_merged.filter(~pl.col('hgvs_pro').is_null())['gene_mutant'].value_counts().filter(pl.col('count')>1).sort("count", descending=True)
reps

In [ ]:
pg_reps = mave_var_merged.filter(pl.col('gene_mutant').is_in(reps['gene_mutant']))
pg_reps

In [ ]:
pg_reps_stats = (
    pg_reps.group_by('gene_mutant')
    .agg([
        pl.count('score').alias('num_reps'),
        pl.mean('score').alias('dms_score_mean'),
        pl.std('score').alias('dms_score_std'),
        pl.min('score').alias('dms_score_min'),
        pl.max('score').alias('dms_score_max')
    ])
).sort('dms_score_std', descending=True)

pg_reps_stats

In [ ]:
plt_df = pg_reps_stats.filter(~pl.col('dms_score_std').is_null())#.filter(pl.col('dms_score_std')<100)


(
    ggplot(plt_df, aes(x='dms_score_mean', y='dms_score_std')) +
    geom_bin2d(bins=100) +
    geom_abline() +
    scale_x_log10() +
    scale_y_log10() +
    theme_bw()
)

### Check for BRCA2 and GCK

In [ ]:
mave_var_merged.filter(~pl.col('hgvs_pro').is_null())['gene_name'].value_counts().sort('count', descending=True).head(10)

## merge with RAP variants

In [ ]:
gb_res = pd.read_parquet('/s/project/deeprvat/ukb_gym/genebass/genebass_all_associations_p_e-5.parquet')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

# gb_res = pl.from_pandas(gb_res.query("(significant == True) & (trait_type=='continuous') & (modifier != 'custom') & ('pLoF' in annotation)"))

gb_res = pl.from_pandas(gb_res.query("(significant == True) & (modifier != 'custom') & ('pLoF' in annotation)"))
gb_res

In [ ]:
gb_res.filter(pl.col('gene_symbol') == 'TP53')

In [ ]:
gb_rap = pl.read_parquet('/s/project/deeprvat/ukb_gym/genebass/250709_rap_prs_phenos_genebass_assocs.parquet')
gb_rap

In [ ]:
# rap_vars = rap_vars.with_columns([
#     # Split `protein_position` on '/' and take the first part
#     pl.col("protein_position").str.split("/").list.get(0).alias("mutant_position"),
#     pl.col("protein_position").str.split("/").list.get(1).alias("protein_length"),
# ])

# rap_vars = rap_vars.with_columns([
#     # Replace '/' in `Amino_acids` with `mutant_position`
#     (
#         pl.format(
#             "{}{}{}",
#             pl.col("amino_acids").str.split("/").list.get(0),
#             pl.col("mutant_position"),
#             pl.col("amino_acids").str.split("/").list.get(1)
#         )
#     ).alias("mutant")
# ])

# rap_vars.write_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')

rap_vars = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')
rap_vars = rap_vars.filter(pl.col('AF_ukb')<0.001)

rap_vars = rap_vars.filter(pl.col('region').is_in(gb_rap['gene_id'].unique()))
# rap_vars = rap_vars.filter(pl.col('region').is_in(gb_res['gene_id'].unique()))
rap_vars

In [ ]:
mave_tmp = mave_var_merged[['gene_id', 'mutant_short', 'score', 'gene_symbol']].unique().rename({
    'mutant_short': 'mutant',
    'gene_id': 'region'
})
mave_tmp

In [ ]:
mave_var_merged.filter(pl.col('gene_symbol')=='LDLR')

In [ ]:
mj = rap_vars[['region', 'mutant']].unique().join(mave_tmp, on=['region', 'mutant'], how='inner')
mj

In [ ]:
vc = mj['gene_symbol'].value_counts().sort('count', descending=True)
vc

In [ ]:
# Example counts
vc = (
    mj['gene_symbol']
    .value_counts()
    .sort('count', descending=True)
).to_pandas()

# Make gene_symbol a categorical with sorted levels
vc['gene_symbol'] = pd.Categorical(vc['gene_symbol'], categories=vc['gene_symbol'], ordered=True)

# Plot
(
    ggplot(vc, aes(x='gene_symbol', y='count')) +
    geom_col() +
    theme_bw() +
    scale_y_log10() +
    annotation_logticks(sides='l') +
    theme(
        axis_text_x=element_text(angle=90),
        figure_size=(8, 4)
    )
)

In [ ]:
gb_res.filter(pl.col('gene_symbol')=='GCK')

In [ ]:
gb_res.filter(pl.col('gene_symbol')=='CHEK2')

In [ ]:
gb_res.filter(pl.col('gene_symbol')=='CRX')